# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsimaZaheer/Task1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My unit of analysis is one content page for one client on one day. I will use the `fact_content_daily_performance` table for this work.

For development and verification, I will use the mid-panel month of March 2026 (`month = 2026-03`). This keeps the development period away from the final June 2026 month, which should be treated as a future/sealed period when building future-looking labels.

My decision is to prioritize content pages for review based on observable search and engagement signals available at the decision moment.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### Features
- `impressions` — observed search visibility available at the decision moment.
- `clicks` — observed search clicks available at the decision moment.
- `position` — observed average search position available at the decision moment.
- `sessions` — observed analytics sessions available at the decision moment.
- `content_age_days` — content age known at the decision moment.

### Label / proxy
- `trend_direction` — used as a proxy for observed movement in the starter task. It is not a future outcome and will not be treated as proof that a page will decline.

### Context
- `client_hash_id` — used only to group pages and support client-level validation.
- `content_hash_id` — identifies the content item and is used for joins and grouping.
- `report_date` — identifies the observation date and defines the time window.

### Excluded
- Raw URLs, titles, queries, domains, and client names — these are not available in the safe release and must not be reconstructed or published.
- Product decision scores or flags — these should not be used as model features because they represent existing product decisions rather than independent observed signals.
- Future-window measurements — excluded from features because using information from after the decision moment would create leakage.

In [9]:
import os
import duckdb

# Read the token securely from Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection is ready.")

Hugging Face connection is ready.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Check that the warehouse table is accessible
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│   78835655 │
└────────────┘

In [11]:
# Verification Query 1: Check the grain for March 2026

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        client_hash_id || '|' ||
        content_hash_id
    ) AS unique_grain_rows
FROM read_parquet('{rel}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_rows │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [12]:
# Verification Query 2: March 2026 row count and date span

con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{rel}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [13]:
# Verification Query 3: Check data availability

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM read_parquet('{rel}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │ gsc_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │             413966 │            3611061 │
└────────────┴────────────────────┴────────────────────┘

## 4. Data limits

This data has some important limitations. The warehouse is an unbalanced panel, so different clients have different amounts of history. Some early rows may have GSC data but not GA4 data, so missing analytics data should not automatically be treated as zero traffic. My March 2026 slice is useful for development, but it represents only one month and may not capture seasonality or longer-term content changes. For future-looking prediction, feature and target windows must not overlap because that could cause data leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.